# CE1110: Tarea 1 — Apartado 2.2

**Generación y caracterización de tres casos de señal contaminada.** Se usan datos sintéticos, semillas distintas y un script reproducible que crea WAV/NPY, gráficos, metadatos y una tabla de SNR.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

BASE = Path.cwd().resolve()
if not (BASE / "src" / "generacion_senales.py").exists():
    BASE = BASE / "proyecto_grupal_1_reduccion_ruido" / "tarea1"
if not (BASE / "src" / "generacion_senales.py").exists():
    raise FileNotFoundError("Abrir el notebook desde carpeta tarea1 o raíz del repositorio.")
sys.path.insert(0, str(BASE / "src"))
import generacion_senales as g
print("Carpeta de trabajo:", BASE)

## 1. Generar desde cero

Se genera una referencia multitono común; tres realizaciones de ruido independientes y escaladas a la potencia necesaria para las SNR indicadas. La ganancia global contra recorte se aplica uniformemente a todos los casos.

In [ ]:
registros = g.generar(BASE)
config = json.loads((BASE / "metadatos" / "configuraciones.json").read_text(encoding="utf-8"))
print("fs =", config["fs_hz"], "Hz; N_bloque =", config["n_bloque"], "; L =",config["n_total"])
print("Duración total =",config["duracion_s"],"s; ganancia anticlip =",config["ganancia_global_anticlip"])

## 2. Validar matemáticamente SNR, energía y señales

In [ ]:
datos = pd.read_csv(BASE / "resultados" / "metricas_generacion.csv")
display(datos[["caso","semilla","fs_hz","n_bloque","n_total","snr_objetivo_db","snr_obtenida_db","error_snr_db","pico_contaminada"]])
assert (np.abs(datos["error_snr_db"]) < 1e-10).all()
for k in (1,2,3):
    p = BASE / "pruebas" / f"caso_{k}"
    s, r, x = [np.load(p / f"{nom}.npy") for nom in ("limpia", "ruido", "contaminada")]
    assert np.allclose(x, s+r, rtol=0, atol=1e-12)
    assert abs(g.energia_por_bandas(x).sum() - 100) < 1e-10
print("Validación de los tres casos superada.")

## 3. Cuatro representaciones por caso

Tiempo, magnitud FFT, fase de componentes significativas y porcentaje de energía por bandas. La figura de PSD estimada por Welch complementa la caracterización del ruido.

In [ ]:
for k in (1,2,3):
    print(f"CASO {k}: RESUMEN")
    display(Image(filename=str(BASE / "resultados" / "figuras" / f"caso{k}_resumen.png"), width=900))
    print(f"CASO {k}: DENSIDAD ESPECTRAL DE POTENCIA")
    display(Image(filename=str(BASE / "resultados" / "figuras" / f"caso{k}_psd.png"), width=740))

## 4. Caracterización y limitaciones

- **Caso 1:** interferencia coherente muy estrecha a 1792 Hz, separada de la banda con los tres tonos útiles. Su pico puede detectarse por contraste respecto a la mediana del espectro, pero no basta para saber si es interferencia sin información previa.
- **Caso 2:** ruido blanco aleatorio con energía distribuida ampliamente; su fase no sigue una relación estable entre bins. Puede estimarse la densidad de ruido de fondo con Welch o percentiles robustos.
- **Caso 3:** energía interferente mayormente entre 850 y 1300 Hz, coincidente con el tono de 1024 Hz. Sin referencia independiente, dos fuentes compartiendo la frecuencia no pueden separarse exactamente a partir de una sola mezcla. El rechazo reduce ruido y también puede eliminar parte de la componente útil.